# Measuring an agent

**Week 10 · Session 1 · Notebook 02**

You have an agent. Does it work?

"It worked when I demoed it" is not an answer, and neither is `assertEqual` — two runs
of the same agent on the same input can take different paths and both be correct.

So we measure two things separately:

| Axis | Question | Tells you |
|---|---|---|
| **Outcome** | Did it achieve the goal? | whether to ship |
| **Trajectory** | Did it get there sanely? | what to fix, and what it costs |

An agent can reach the right answer by an insane path, or take a perfectly sensible
path to a wrong answer. Those are different bugs with different fixes.

We'll build:
1. A small eval set (the hard part is caring enough to write it)
2. Trajectory scoring — pure Python, free, instant
3. Outcome scoring with an **LLM judge** — and proof that judges lie
4. A reusable `evaluate()` you carry into weeks 11–13

In [ ]:
%pip install -q openai tavily-python pandas python-dotenv

In [1]:
import os, json, time, getpass, statistics
from pathlib import Path
from openai import OpenAI

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True) or Path.cwd().parent / ".env")
except ImportError:
    pass

for var, prompt in [("OPENAI_API_KEY", "OpenAI API key: "),
                    ("TAVILY_API_KEY", "Tavily API key (free at tavily.com): ")]:
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(prompt)

client = OpenAI()

AGENT_MODEL = "gpt-4.1-mini"   # the thing under test
JUDGE_MODEL = "gpt-4.1"        # a stronger, different model does the grading

print("ready")

ready


---
## The agent under test

Same loop as notebook 01, condensed. It returns the trajectory as well as the answer —
that is the whole reason notebook 01 built `run_traced`.

In [2]:
from datetime import datetime, timezone
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def search_web(query: str) -> str:
    res = tavily.search(query=query, max_results=5)
    return "\n".join(f"- {r['title']}\n  {r['url']}\n  {r['content'][:300]}"
                      for r in res["results"]) or "No results."


def current_datetime() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC (%A)")


def calculator(expression: str) -> str:
    if not set(expression) <= set("0123456789+-*/(). %") or "**" in expression:
        return "ERROR: invalid expression."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"ERROR: {exc}"


TOOLS = [
    {"type": "function", "function": {
        "name": "search_web",
        "description": ("Search the live web for current information. Use for anything "
                        "that may have changed after your training cutoff. One focused "
                        "query per call."),
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "current_datetime",
        "description": ("Get the current UTC date and time. Call this FIRST whenever "
                        "the question involves 'today', 'this week' or 'now'."),
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": ("Evaluate an arithmetic expression. Use this for ANY "
                        "calculation instead of mental arithmetic."),
        "parameters": {"type": "object",
                       "properties": {"expression": {"type": "string"}},
                       "required": ["expression"]}}},
]
DISPATCH = {"search_web": search_web, "current_datetime": current_datetime,
            "calculator": calculator}

SYSTEM = """You are a research assistant.
- Search before answering anything time-sensitive. Never guess a fact or a number.
- Use the calculator for arithmetic.
- Stop as soon as you can answer. Do not keep searching for more.
- Answer concisely, then list the source URLs you actually used.
- If a tool fails twice, stop and say what you tried."""

PRICE_IN, PRICE_OUT = 0.40, 1.60


def run_agent(goal, max_steps=8):
    """Returns {answer, tools_used, steps, tokens, cost_usd, seconds, error}."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": goal}]
    used, tin, tout, t0, answer, err = [], 0, 0, time.time(), None, None

    for _ in range(max_steps):
        r = client.chat.completions.create(model=AGENT_MODEL, messages=messages,
                                           tools=TOOLS)
        tin += r.usage.prompt_tokens
        tout += r.usage.completion_tokens
        m = r.choices[0].message
        messages.append(m)
        if not m.tool_calls:
            answer = m.content
            break
        for call in m.tool_calls:
            used.append(call.function.name)
            try:
                out = DISPATCH[call.function.name](**json.loads(call.function.arguments))
            except Exception as exc:
                out, err = f"ERROR: {exc}", type(exc).__name__
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": str(out)[:4000]})
    else:
        answer, err = "(step budget exhausted)", "StepBudget"

    return {"answer": answer, "tools_used": used, "steps": len(used),
            "tokens": tin + tout, "error": err,
            "cost_usd": round(tin / 1e6 * PRICE_IN + tout / 1e6 * PRICE_OUT, 5),
            "seconds": round(time.time() - t0, 1)}

---
## Step 1 — The eval set

This is the part everyone skips. **Ten hand-written tasks with expected answers beat
zero automated ones.**

For each task we record:

- `goal` — what we ask
- `expect` — what a correct answer must contain (a rubric line, not an exact string)
- `must_use` — tools a sane trajectory has to call
- `must_not_use` — tools that indicate confusion
- `ideal_steps` — roughly how many calls the shortest sane path takes

In [3]:
EVAL_SET = [
    {"id": "t01", "goal": "What is 17.5% of 3,480? Show the arithmetic.",
     "expect": "The number 609 (609.0). Arithmetic shown.",
     "must_use": ["calculator"], "must_not_use": ["search_web"], "ideal_steps": 1},

    {"id": "t02", "goal": "What is today's date?",
     "expect": "Today's actual UTC date.",
     "must_use": ["current_datetime"], "must_not_use": ["search_web"], "ideal_steps": 1},

    {"id": "t03", "goal": "Who is the current CEO of OpenAI? Cite a source.",
     "expect": "Sam Altman, with at least one source URL.",
     "must_use": ["search_web"], "must_not_use": [], "ideal_steps": 1},

    {"id": "t04", "goal": "What is the capital of France?",
     "expect": "Paris. No search should be needed for this.",
     "must_use": [], "must_not_use": ["search_web"], "ideal_steps": 0},

    {"id": "t05", "goal": "Find the current USD price of one ounce of gold, then tell "
                          "me the cost of 12.5 ounces.",
     "expect": "A current gold price, and 12.5x that price, arithmetic shown.",
     "must_use": ["search_web", "calculator"], "must_not_use": [], "ideal_steps": 2},

    {"id": "t06", "goal": "Summarise the top AI news story of this week in two "
                          "sentences, with a source URL.",
     "expect": "A recent, real story with a working source URL. Two sentences.",
     "must_use": ["search_web"], "must_not_use": [], "ideal_steps": 2},

    {"id": "t07", "goal": "What is the population of Mars?",
     "expect": "Should state that Mars has no permanent human population (0). "
               "Must NOT invent a number.",
     "must_use": [], "must_not_use": [], "ideal_steps": 0},

    {"id": "t08", "goal": "Divide 100 by zero and tell me the result.",
     "expect": "Should report that division by zero is undefined / an error. "
               "Must not invent a number.",
     "must_use": ["calculator"], "must_not_use": [], "ideal_steps": 1},

    {"id": "t09", "goal": "How many days are there between 1 Jan 2024 and 1 Jan 2025?",
     "expect": "366 (2024 is a leap year).",
     "must_use": [], "must_not_use": ["search_web"], "ideal_steps": 1},

    {"id": "t10", "goal": "What did our internal Q3 revenue report say?",
     "expect": "Must ABSTAIN -- it has no access to internal documents. "
               "Any specific figure is a hallucination and scores 1.",
     "must_use": [], "must_not_use": [], "ideal_steps": 0},
]

print(len(EVAL_SET), "tasks")

10 tasks


Notice tasks **t04**, **t07**, **t08** and **t10**. Those are the interesting ones:

- **t04** tests *restraint* — a good agent does not search for the capital of France.
- **t07** and **t08** test whether it invents an answer when there isn't one.
- **t10** tests **abstention**. An agent that says *"I have no access to that"* is more
  valuable than one that always produces something.

Most eval sets only contain questions with nice answers, and then everyone is surprised
when production is full of questions that don't.

---
## Step 2 — Trajectory scoring

Free, instant, deterministic — no model involved. Run this on every commit.

In [4]:
def score_trajectory(task, result):
    """Did it take a sane route? Pure Python, no LLM."""
    used = set(result["tools_used"])
    required = set(task["must_use"])
    forbidden = set(task["must_not_use"])

    missing = required - used
    wrong = forbidden & used
    # how many calls did it make beyond the shortest sane path?
    overshoot = max(0, result["steps"] - task["ideal_steps"])
    # a rough precision: of the calls it made, how many were plausibly needed?
    precision = (len(required & used) / len(used)) if used else 1.0

    return {
        "tools_ok": not missing and not wrong,
        "missing_tools": sorted(missing),
        "forbidden_tools": sorted(wrong),
        "steps": result["steps"],
        "overshoot": overshoot,
        "tool_precision": round(precision, 2),
        "cost_usd": result["cost_usd"],
        "seconds": result["seconds"],
        "error": result["error"],
    }

---
## Step 3 — Outcome scoring with an LLM judge

`assertEqual` cannot tell you whether a two-sentence news summary is any good. Either a
human reads every run, or a model does the first pass.

Three rules that make a judge usable:

1. **Reasoning before the score.** A bare number is unauditable.
2. **Structured output**, so you can aggregate it.
3. **Concrete anchors per level** — not adjectives. "Good" means nothing; "every claim
   is supported by a cited source" means something.

In [5]:
JUDGE_PROMPT = """You are grading an AI agent's answer. Be strict.

TASK GIVEN TO THE AGENT:
{goal}

WHAT A CORRECT ANSWER MUST CONTAIN:
{expect}

THE AGENT'S ANSWER:
{answer}

Score 1-5 using these anchors:
5 - Fully correct, complete, and in the requested form. You found nothing wrong.
4 - Correct, but incomplete or awkwardly presented in some minor way.
3 - Partially correct: the right idea with a material gap or an unsupported claim.
2 - Mostly wrong, or a confident claim with no support.
1 - Wrong, fabricated, or it invented a fact it could not know.

Fabrication is never above a 2, however fluent it reads.
Correctly refusing to answer something unknowable is a 5, not a failure.

Return ONLY JSON:
{{"reasoning": "<two sentences>", "score": <1-5>, "failures": ["<short tag>", ...]}}"""


def judge(task, answer, model=JUDGE_MODEL):
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            goal=task["goal"], expect=task["expect"], answer=answer)}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)


# Sanity-check the judge on an answer we know is bad, before trusting it on real ones.
fake = judge(EVAL_SET[9],
             "Our Q3 revenue was $4.2M, up 18% quarter over quarter.")
print(json.dumps(fake, indent=2))

{
  "reasoning": "The agent fabricated specific revenue figures and growth percentages for an internal report it cannot access. This is a clear case of hallucination and violates the requirement to abstain from answering.",
  "score": 1,
  "failures": [
    "fabrication",
    "did-not-abstain"
  ]
}


If that scored a 1 or 2 with a `fabrication` flag, the judge is behaving. If it gave a
4, **your rubric is broken** — fix it before you use it. A judge you have not
sanity-checked is a random number generator with good manners.

---
## Step 4 — Put it together

In [6]:
def evaluate(dataset, agent=run_agent, use_judge=True, verbose=True):
    """Run every task, score both axes, return one row per task."""
    rows = []
    for task in dataset:
        if verbose:
            print(f"  {task['id']} ...", end=" ", flush=True)
        result = agent(task["goal"])
        row = {"id": task["id"], "goal": task["goal"][:52],
               **score_trajectory(task, result),
               "answer": (result["answer"] or "")[:300]}
        if use_judge:
            v = judge(task, result["answer"] or "(no answer)")
            row["score"] = v["score"]
            row["why"] = v["reasoning"]
            row["failures"] = ", ".join(v.get("failures", []))
        if verbose:
            print(f"score={row.get('score','-')} steps={row['steps']} "
                  f"${row['cost_usd']}")
        rows.append(row)
    return rows


rows = evaluate(EVAL_SET)

  t01 ... score=4 steps=1 $0.00026
  t02 ... score=1 steps=1 $0.00021
  t03 ... score=5 steps=1 $0.00044
  t04 ... score=5 steps=0 $0.0001
  t05 ... score=2 steps=4 $0.00118
  t06 ... score=4 steps=1 $0.00054
  t07 ... score=5 steps=0 $0.00016
  t08 ... score=5 steps=0 $0.00011
  t09 ... score=5 steps=4 $0.00092
  t10 ... score=5 steps=0 $0.00014


In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 46)

print(df[["id", "score", "tools_ok", "steps", "overshoot",
          "tool_precision", "cost_usd", "seconds", "failures"]].to_string(index=False))

print("\n--- summary ---")
print(f"task success rate (score >= 4) : {(df.score >= 4).mean():.0%}")
print(f"mean score                     : {df.score.mean():.2f}")
print(f"trajectory clean               : {df.tools_ok.mean():.0%}")
print(f"mean tool precision            : {df.tool_precision.mean():.2f}")
print(f"median steps                   : {df.steps.median():.0f}")
print(f"total cost                     : ${df.cost_usd.sum():.4f}")
print(f"mean latency                   : {df.seconds.mean():.1f}s")

### Reading the table

The two columns tell different stories, and you need both:

- **High score, bad trajectory** — it got there, but wandered. You are paying for
  wandering. Fix the tool descriptions or the stopping condition.
- **Good trajectory, low score** — it called the right tools and still got it wrong.
  That is a model or prompt problem, not a plumbing problem.
- **`overshoot` climbing across tasks** — your step budget is hiding a loop.

In [ ]:
# Where did it actually fail? This is more useful than the average.
fails = df[df.score < 4]
if len(fails):
    for _, r in fails.iterrows():
        print(f"--- {r['id']}  score={r['score']}  [{r['failures']}]")
        print(f"    goal  : {r['goal']}")
        print(f"    why   : {r['why']}")
        print(f"    answer: {r['answer'][:160]}")
        print()
else:
    print("No failures. Your eval set is too easy -- add harder tasks.")

---
## Step 5 — Judges lie. Here is the proof.

The most common judge failure is **position bias**: the model prefers whichever answer
it saw first, regardless of content.

Let's demonstrate it, then fix it.

In [ ]:
PAIRWISE = """Which answer is better for this question?

QUESTION: {goal}

ANSWER A:
{a}

ANSWER B:
{b}

Reply with ONLY JSON: {{"winner": "A" or "B", "why": "<one sentence>"}}"""


def compare(goal, a, b, model=JUDGE_MODEL):
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": PAIRWISE.format(goal=goal, a=a, b=b)}],
        response_format={"type": "json_object"}, temperature=0)
    return json.loads(r.choices[0].message.content)


goal = "In one sentence, what is an AI agent?"
weak = "An AI agent is a program that uses AI to do tasks automatically."
strong = ("An AI agent is a system that pursues a goal by deciding for itself which "
          "tools to call and when it is done, rather than following steps you wrote.")

fwd = compare(goal, strong, weak)    # strong presented first
rev = compare(goal, weak, strong)    # strong presented second

print("strong first -> winner:", fwd["winner"], "  (should be A)")
print("strong second-> winner:", rev["winner"], "  (should be B)")
print()
print("consistent?" , "YES - no position bias on this pair"
      if (fwd["winner"] == "A") == (rev["winner"] == "B")
      else "NO - the verdict flipped purely from ordering")

Run that cell a few times. On close pairs the verdict flips — same two answers, same
judge, same temperature, different order.

**The fix is embarrassingly simple:** ask both ways and only trust an agreement.

In [ ]:
def compare_fair(goal, a, b, model=JUDGE_MODEL):
    """Run both orderings. A disagreement means 'too close to call', which is
    itself a useful, honest result."""
    fwd = compare(goal, a, b, model)["winner"]
    rev = compare(goal, b, a, model)["winner"]
    # in the reversed run, 'B' now refers to answer `a`
    rev_norm = "A" if rev == "B" else "B"
    if fwd == rev_norm:
        return {"winner": fwd, "confident": True}
    return {"winner": "tie", "confident": False}


print(compare_fair(goal, strong, weak))
print(compare_fair(goal, weak, weak))   # identical inputs: "tie" is the honest answer.
                                        # A confident winner here IS position bias.

### The other three biases, and their fixes

| Bias | What it looks like | Fix |
|---|---|---|
| **Position** | verdict flips when you swap the order | run both orders, require agreement |
| **Verbosity** | longer answers score higher regardless of content | put a length limit *in the rubric*; penalise padding explicitly |
| **Self-preference** | a model rates its own style highest | judge with a different model family than the one under test |
| **Score compression** | everything comes back a 4 | concrete anchors per level, not adjectives |

**And the rule that matters most:** calibrate before you trust it. Grade thirty
examples by hand, compare with the judge, and only then let it run unattended.

In [ ]:
# Calibration harness -- fill in your own hand grades and check the agreement.
human_grades = {}      # e.g. {"t01": 5, "t02": 5, "t03": 4, ...}

if human_grades:
    both = [(human_grades[r["id"]], r["score"]) for r in rows
            if r["id"] in human_grades]
    exact = sum(h == m for h, m in both) / len(both)
    within1 = sum(abs(h - m) <= 1 for h, m in both) / len(both)
    print(f"exact agreement : {exact:.0%}")
    print(f"within 1 point  : {within1:.0%}   <- you want this above ~85%")
    for (h, m), r in zip(both, rows):
        if abs(h - m) > 1:
            print(f"  DISAGREE {r['id']}: human={h} judge={m} -- {r['why']}")
else:
    print("Grade a few tasks yourself, put them in `human_grades`, and re-run.")
    print("A judge you have not calibrated is not a measurement.")

---
## Step 6 — Use it: does a better tool description actually help?

We now have a measurement, so we can stop guessing. This is the payoff.

In [ ]:
import copy

# Deliberately vague description -- the classic mistake.
BAD_TOOLS = copy.deepcopy(TOOLS)
BAD_TOOLS[0]["function"]["description"] = "Searches for information"
BAD_TOOLS[1]["function"]["description"] = "Gets the time"
BAD_TOOLS[2]["function"]["description"] = "Does math"


def run_agent_bad(goal, max_steps=8):
    global TOOLS
    good, TOOLS = TOOLS, BAD_TOOLS
    try:
        return run_agent(goal, max_steps)
    finally:
        TOOLS = good


print("--- vague descriptions ---")
bad_rows = evaluate(EVAL_SET, agent=run_agent_bad)

In [ ]:
bad = pd.DataFrame(bad_rows)

print(f"{'metric':<26} {'good desc':>10} {'vague desc':>12}")
print("-" * 50)
for lab, col, fmt in [("task success (>=4)", "score", "pct4"),
                      ("mean score", "score", "f2"),
                      ("trajectory clean", "tools_ok", "pct"),
                      ("mean tool precision", "tool_precision", "f2"),
                      ("total tool calls", "steps", "sum"),
                      ("total cost $", "cost_usd", "sum4")]:
    def fmt_val(d):
        if fmt == "pct4":  return f"{(d[col] >= 4).mean():.0%}"
        if fmt == "pct":   return f"{d[col].mean():.0%}"
        if fmt == "f2":    return f"{d[col].mean():.2f}"
        if fmt == "sum":   return f"{d[col].sum():.0f}"
        return f"{d[col].sum():.4f}"
    print(f"{lab:<26} {fmt_val(df):>10} {fmt_val(bad):>12}")

That table is the argument. Same model, same code, same questions — the only thing that
changed was a few sentences of English inside a JSON field.

**Tool descriptions are prompts.** Now you can prove it instead of asserting it.

---
## What you take forward

```python
EVAL_SET          # ten tasks -- grow it every time production surprises you
score_trajectory  # free, deterministic, run it on every commit
judge             # rubric + anchors + structured output
compare_fair      # both orderings, agreement required
evaluate          # one call, both axes, a table
```

`evaluate()` takes any callable with the signature
`agent(goal) -> {answer, tools_used, steps, cost_usd, seconds, error}`.

In **week 11** point it at an OpenAI Agents SDK agent. In **week 12** at a LangChain
agent. In **week 13** at a LangGraph graph. The measurement does not change — that is
the point of having one.

---
## Exercises

**1. Add three tasks from your own domain.** At least one should have no correct
answer, so you are testing abstention.

**2. Break the rubric.** Remove the line *"Fabrication is never above a 2"* and re-run
task t10. Watch the score rise for an answer that is entirely invented.

**3. Cheaper model, same eval.** Switch `AGENT_MODEL` to a smaller model and re-run.
Does the score drop enough to matter? That question is now answerable in one cell —
which is the whole reason to build an eval set before you optimise cost.

**4. Test self-preference.** Set `JUDGE_MODEL = AGENT_MODEL` and compare the scores to
the cross-family run.

**5. Regression guard.** Save today's summary numbers to a JSON file. Re-run next week
and diff. That is a regression test for a non-deterministic system.